In [1]:
import time
import matplotlib
import numpy as np

matplotlib.use("Agg")

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

In [2]:
Z_raw = np.load("./data/fuji_elevation.npy").astype(np.float64)
lat   = np.load("./data/fuji_lat.npy")
lon   = np.load("./data/fuji_lon.npy")

# .npy хранят данные в порядке «север -Ю юг»,
# переворачиваем, чтобы y возрастал с юга на север
Z_raw  = Z_raw[::-1, :]
lat_sn = lat[::-1]

lat0 = 35.3606
deg2m_lat = 111_320.0
deg2m_lon = 111_320.0 * np.cos(np.radians(lat0))

x_m = (lon    - lon[0])    * deg2m_lon
y_m = (lat_sn - lat_sn[0]) * deg2m_lat

dx = x_m[1] - x_m[0]
dy = y_m[1] - y_m[0]
nrows, ncols = Z_raw.shape

print(f"\nСетка: {nrows}x{ncols}")
print(f"Шаг: dx = {dx:.1f} м, dy = {dy:.1f} м")
print(f"Область: {x_m[-1]/1000:.1f} x {y_m[-1]/1000:.1f} км")
print(f"Высота: {Z_raw.min():.0f} — {Z_raw.max():.0f} м")


Сетка: 1080x1080
Шаг: dx = 25.2 м, dy = 30.9 м
Область: 27.2 x 33.4 км
Высота: 63 — 3759 м


In [4]:
# ══════════════════════════════════════════════
# 2. Эталонный объём (2D трапеции)
# ══════════════════════════════════════════════

def volume_trapezoid_2d(Z, dx, dy, h0):
    """
    Объём над уровнем h0 по формуле 2D-трапеций на регулярной сетке.
    """
    H = np.maximum(Z - h0, 0)

    # Веса: углы=1/4, края=1/2, внутренние=1
    W = np.ones_like(H)
    W[0, :] *= 0.5;  W[-1, :] *= 0.5
    W[:, 0] *= 0.5;  W[:, -1] *= 0.5

    return dx * dy * np.sum(W * H)


h0 = 500.0  # базовый уровень (м)

V_ref = volume_trapezoid_2d(Z_raw, dx, dy, h0)
print(f"\n--- Эталонный объём (h₀ = {h0:.0f} м) ---")
print(f"V_ref = {V_ref:.6e} м³ = {V_ref/1e9:.3f} км³")


--- Эталонный объём (h₀ = 500 м) ---
V_ref = 5.134853e+11 м³ = 513.485 км³
